# Streaming Dataset with IterableDataset — Solution

In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader, get_worker_info
import os
import tempfile

torch.manual_seed(42)
N, D = 10_000, 5
X = torch.randn(N, D)
y = X @ torch.randn(D) + 0.1 * torch.randn(N)

data_path = os.path.join(tempfile.gettempdir(), "streaming_test_data.csv")
with open(data_path, "w") as f:
    for i in range(N):
        row = ",".join(f"{v:.6f}" for v in X[i].tolist()) + f",{y[i]:.6f}\n"
        f.write(row)

In [ ]:
class StreamingDataset(IterableDataset):
    """Streams (x, y) pairs from a CSV file, one line at a time."""

    def __init__(self, filepath: str):
        super().__init__()
        self.filepath = filepath

    def __iter__(self):
        # In the main process get_worker_info() is None: one "worker" that reads everything.
        # Inside a DataLoader worker it says which of the num_workers copies this is.
        worker_info = get_worker_info()
        worker_id = 0 if worker_info is None else worker_info.id
        num_workers = 1 if worker_info is None else worker_info.num_workers

        # Open the file here, not in __init__: every epoch (and every worker) needs its own handle
        with open(self.filepath) as f:
            for line_idx, line in enumerate(f):  # the file object is a lazy line iterator
                if line_idx % num_workers != worker_id:
                    continue  # another worker owns this line
                if not line.strip():
                    continue
                values = [float(v) for v in line.strip().split(",")]
                x = torch.tensor(values[:-1])
                y = torch.tensor(values[-1])
                yield x, y


# Quick demo
ds = StreamingDataset(data_path)
loader = DataLoader(ds, batch_size=32)
bx, by = next(iter(loader))
print(f"Batch: X={bx.shape}, y={by.shape}")

In [ ]:
# === Tests ===

# Test 1: Single-process iteration yields all samples
ds = StreamingDataset(data_path)
loader = DataLoader(ds, batch_size=64)
total = sum(by.shape[0] for _, by in loader)
assert total == N, f"Expected {N} samples, got {total}"
print(f"PASSED: Single-process yields all {N} samples")

# Test 2: Correct shapes
ds2 = StreamingDataset(data_path)
loader2 = DataLoader(ds2, batch_size=32)
bx, by = next(iter(loader2))
assert bx.shape == (32, D), f"Expected X shape (32, {D}), got {bx.shape}"
assert by.shape == (32,), f"Expected y shape (32,), got {by.shape}"
print("PASSED: Correct batch shapes")

# Test 3: Values match original data
ds3 = StreamingDataset(data_path)
loader3 = DataLoader(ds3, batch_size=N)
bx_all, by_all = next(iter(loader3))
assert torch.allclose(bx_all, X, atol=1e-4), "X values don't match"
assert torch.allclose(by_all, y, atol=1e-4), "y values don't match"
print("PASSED: Streamed values match original tensors")

# Test 4: Worker sharding. Inside a DataLoader worker, get_worker_info() reports the worker's
# id; we fake that context here (a private hook) so the test runs on every OS.
from contextlib import contextmanager
from torch.utils.data._utils import worker as _worker

@contextmanager
def as_worker(worker_id, num_workers, dataset):
    _worker._worker_info = _worker.WorkerInfo(id=worker_id, num_workers=num_workers, seed=0, dataset=dataset)
    try:
        yield
    finally:
        _worker._worker_info = None

for num_workers in (2, 3):
    shards = []
    for worker_id in range(num_workers):
        ds4 = StreamingDataset(data_path)
        with as_worker(worker_id, num_workers, ds4):
            shards.append(torch.stack([sy for _, sy in ds4]))
    sizes = [len(s) for s in shards]
    assert sum(sizes) == N, \
        f"{num_workers} workers yielded {sum(sizes)} samples in total, expected {N} — rows are duplicated or lost"
    assert torch.allclose(torch.cat(shards).sort().values, y.sort().values, atol=1e-4), \
        "Together the shards must contain every row exactly once"
    assert min(sizes) > N / (2 * num_workers), f"Shards are very unbalanced: {sizes}"
print("PASSED: Each simulated worker gets a disjoint shard; together they cover the file")

# Test 5: The real thing - two worker processes (Linux only: needs the "fork" start method,
# which, unlike spawn, does not pickle the notebook-defined class)
import sys
if sys.platform.startswith("linux"):
    loader5 = DataLoader(StreamingDataset(data_path), batch_size=64, num_workers=2,
                         multiprocessing_context="fork")
    y5 = torch.cat([by for _, by in loader5])
    assert len(y5) == N, f"With 2 workers got {len(y5)} samples, expected {N} — are the workers duplicating rows?"
    assert torch.allclose(y5.sort().values, y.sort().values, atol=1e-4), "Multi-worker loading lost or duplicated rows"
    print("PASSED: DataLoader(num_workers=2) yields every row exactly once")
else:
    print("SKIPPED: real multi-worker test (needs Linux)")

print("\nAll tests passed!")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("streaming-dataset", StreamingDataset=StreamingDataset)

### Notes for the interview

- **Every worker still reads the whole file** with modulo sharding; it just skips the lines it does not own. That is fine for a CSV like this, but for large files shard by **byte range** (seek to `k * size / n`, skip to the next newline, stop after `(k + 1) * size / n`) or, better, pre-split the data into many files and assign whole files to workers — the approach WebDataset and Mosaic Streaming take.
- **Distributed training adds a second level of sharding.** With DDP every rank also runs its own DataLoader workers, so shard over `world_size * num_workers` readers with id `rank * num_workers + worker_id`; otherwise every rank sees the same data.
- **Ordering.** Each worker forms its own batches, and the DataLoader takes batches from the workers in round-robin order, so the global order is not file order and there can be up to `num_workers` partial batches at the end of an epoch.
- **Re-iterability.** Opening the file inside `__iter__` (not `__init__`) is what makes a second epoch work: a file handle opened once would already be exhausted.